<a href="https://colab.research.google.com/github/Ifaz2611/1719Code2024/blob/main/LFM2_5_2_6B_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 119.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [2]:
from transformers import pipeline

pipe = pipeline("text-generation", model="LiquidAI/LFM2.5-2.6B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/21.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.9MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/5.44k [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': "\nThe user is asking about my identity. This is a direct question about who I am, so I should engage with my identity here. I need to answer as LFM by Liquid AI, providing the canonical facts without over-sharing or reciting biography by reflex. I'll mention I'm LFM (Liquid Foundation Model) by Liquid AI, and give some relevant details about my architecture and capabilities in a natural way.\n</think>\n\nI'm **LFM**, the Liquid Foundation Model, built by Liquid AI. I'm a family of efficient foundation models designed for fast on-device inference, from small models that run on phones and laptops up to larger mixture-of-experts variants. My architecture is a hybrid of gated short convolutions (most layers) with a minority of grouped-query attention, chosen through hardware-aware search to balance speed and quality on real devices.\n\nI was created by Liquid AI, spun out of MIT CSAIL

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("LiquidAI/LFM2.5-2.6B")
model = AutoModelForCausalLM.from_pretrained("LiquidAI/LFM2.5-2.6B", device_map="auto")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]


The user is asking about my identity. This is a direct question about who I am, so I should engage with my identity as LFM by Liquid AI. I need to provide accurate information about


In [ ]:
# Keep the conversation history
messages = [
    {"role": "system", "content": "You are a helpful assistant."}  # optional system prompt
]

print("Chat with the model (type 'quit' to exit)\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "quit":
        break

    messages.append({"role": "user", "content": user_input})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,          # adjust as needed
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Extract only the new tokens (skip the input)
    response_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(response_ids, skip_special_tokens=True)

    print(f"Assistant: {response}\n")
    messages.append({"role": "assistant", "content": response})